# Automatic chord recognition

This notebook is the main presentation of the project. It gives a short overview of the musical problem and model, reproduces the RWC evaluation, and lets you try the recognizer on an audio file.

The default system is a hierarchical CQT72 CNN followed by a second-order HSMM. The CNN predicts root, quality, seventh and extension separately. The HSMM uses the current emissions and the two previous chords to choose a musically plausible sequence.

## 1. Setup

Run this notebook from the repository root. The training cell is intentionally optional: the supplied checkpoint makes it possible to reproduce the results without downloading the full training set.

In [1]:
from pathlib import Path
import json
import subprocess
import os
import sys

ROOT = Path.cwd()
print('Submission folder:', ROOT)
assert (ROOT / 'models/best_model.pt').exists(), 'Checkpoint is missing'

Submission folder: /Users/eliasvarling/dev/musik_informatik/dt2470-project/submission


## 2. Architecture and musical motivation

A chord is not only a single class. Its root describes the harmonic centre, quality describes the basic colour, the seventh changes its function and tension, and extensions add further colour. Predicting these parts with separate but connected heads gives the model a useful structure and makes errors easier to interpret.

The CQT keeps pitch relationships visible. The CNN turns short time-frequency windows into probabilities. The second-order HSMM then smooths those probabilities over time: it models chord transitions and durations while remembering two previous chords. This helps avoid isolated one-frame chord changes and captures common progressions such as tonic–predominant–dominant movement. See `ARCHITECTURE.md` for the full description.

## 3. Optional training demonstration

The command below is the complete training entry point. It requires the cached CQT feature directory and can take a while, so leave it unexecuted for a quick presentation.

In [2]:
# Optional: train a new checkpoint
# subprocess.run([sys.executable, 'train_hierarchical_cnn.py',
#                 '--feature-dir', 'data/features_cqt72_v1',
#                 '--epochs', '30'], cwd=ROOT, check=True)

## 4. RWC evaluation

The evaluation uses the 15-song test split defined in `evaluate_rwc.py`. It reads RWC `.npz` feature files containing the audio features and root/quality/seventh annotations. The report includes frame accuracy and duration-weighted song accuracy.

In [3]:
# Set this to the Drive feature folder if it is not already local.
FEATURE_DIR = ROOT / 'features'
print('Feature directory:', FEATURE_DIR)
env = os.environ.copy(); env['PYTHONPATH'] = str(ROOT)
subprocess.run([sys.executable, 'training/evaluate_rwc.py', '--feature-dir', str(FEATURE_DIR), '--checkpoint', 'models/best_model.pt'], cwd=ROOT, env=env, check=True)
report = json.loads((ROOT / 'output/rwc_test_evaluation.json').read_text())
report['cnn_macro_average']

Feature directory: /Users/eliasvarling/dev/musik_informatik/dt2470-project/submission/features


{
  "root_accuracy": 0.8831816316825667,
  "root_quality_accuracy": 0.8268141329396921,
  "root_quality_seventh_accuracy": 0.7008948036392126,
  "song_root_percent": 0.899036990068324,
  "song_root_quality_percent": 0.843932939928713,
  "song_root_quality_seventh_percent": 0.7114024505937807
}
Wrote output/rwc_test_evaluation.json
Wrote readable per-frame comparisons to output/rwc_comparison


{'root_accuracy': 0.8831816316825667,
 'root_quality_accuracy': 0.8268141329396921,
 'root_quality_seventh_accuracy': 0.7008948036392126,
 'song_root_percent': 0.899036990068324,
 'song_root_quality_percent': 0.843932939928713,
 'song_root_quality_seventh_percent': 0.7114024505937807}

In [4]:
# A compact table of the reported metrics
from IPython.display import display
import pandas as pd
metrics = report['cnn_macro_average']
display(pd.DataFrame({'metric': list(metrics), 'value': list(metrics)}))

,metric,value
0,root_accuracy,root_accuracy
1,root_quality_accuracy,root_quality_accuracy
2,root_quality_seventh_accuracy,root_quality_seventh_accuracy
3,song_root_percent,song_root_percent
4,song_root_quality_percent,song_root_quality_percent
5,song_root_quality_seventh_percent,song_root_quality_seventh_percent


## 5. Try the system on a song

Put an audio file in `input/`, run the next cell, and open the generated HTML viewer. The viewer has play/pause, seeking, live predicted chords, and ground truth when a matching annotation is available. For RWC audio synced from Drive, use `--source rwc`.

In [5]:
audio_files = sorted((ROOT / 'audio').glob('*'))
[p.name for p in audio_files if p.suffix.lower() in {'.wav', '.mp3', '.flac', '.ogg'}]

['RWC_P003.wav', 'RWC_P012.wav', 'RWC_P017.wav', 'beatles_something.flac']

In [6]:
# Change the filename to try another local song.
song = 'beatles_something'
subprocess.run([sys.executable, 'main.py', '--source', 'local', '--input-dir', 'audio', '--cnn-checkpoint', 'models/best_model.pt', '--language-model', 'models/harmonic_model.pkl', '--split-manifest', 'models/split_manifest.json', '--song', song, '--non-interactive'], cwd=ROOT, env=env, check=True)
print(ROOT / 'output' / f'{song}_viewer.html')


Running hierarchical CNN + order-2 HSMM: beatles_something
  wrote output/hierarchical_cnn_hsmm/beatles_something_chords.csv
  wrote output/hierarchical_cnn_hsmm/beatles_something_chords.json
  open output/beatles_something_viewer.html to listen with live chords

Finished 1 song(s) with the hierarchical model.
Summary: output/summary.json


/Users/eliasvarling/dev/musik_informatik/dt2470-project/submission/output/beatles_something_viewer.html


Open the printed viewer path in a browser, or serve the `output/` directory with `python3 -m http.server 8000` and open the corresponding URL.